# VRIGHT BROTHERS: Sparse-Sensor Reconstruction
## Interactive Walkthrough

This notebook demonstrates the classical Reduced-Order Modeling (ROM) pipeline for sparse-sensor reconstruction on the 2D cylinder wake dataset. It performs Proper Orthogonal Decomposition (POD), optimal sensor placement via Q-DEIM, and linear reconstruction using Tikhonov-regularized Gappy POD.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add root directory to path
sys.path.append(os.path.abspath('..'))

from src.rom.svd_pod import compute_pod
from src.rom.qdeim import qdeim_sensor_placement
from src.rom.gappy_pod import gappy_pod_reconstruct
from src.evaluation.metrics import relative_l2_error, compute_vorticity
from src.evaluation.plotting import plot_reconstruction_comparison

### 1. Load Preprocessed Data
We load the standardized training and testing datasets (3 channels: u, v, p on a 64x128 grid).

In [ ]:
data_dir = os.path.join('..', 'data', 'processed')
train_data = np.load(os.path.join(data_dir, 'train.npz'))
test_data = np.load(os.path.join(data_dir, 'test.npz'))

X_train = train_data['data']
X_test = test_data['data']
mesh_X, mesh_Y = train_data['x'], train_data['y']
cylinder_mask = train_data['cylinder_mask']

M_train, C, ny, nx = X_train.shape
N = C * ny * nx
print(f"Training set shape: {X_train.shape} -> N = {N}")

# Flatten spatial dimensions for ROM (Shape: N x M)
X_train_flat = X_train.reshape(M_train, N).T
X_test_flat = X_test.reshape(X_test.shape[0], N).T

### 2. Proper Orthogonal Decomposition (POD)
We extract the dominant spatial modes using Thin SVD.

In [ ]:
print("Computing POD...")
Phi, S, r, cum_energy = compute_pod(X_train_flat, energy_threshold=0.95)

plt.figure(figsize=(6, 4))
plt.plot(cum_energy[:50], marker='o')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Energy')
plt.xlabel('Number of Modes')
plt.ylabel('Cumulative Energy')
plt.title('POD Energy Spectrum')
plt.legend()
plt.grid(True)
plt.show()

### 3. Optimal Sensor Placement (Q-DEIM)
We select $p=16$ optimal physical sensor locations based on the dominant POD modes.

In [ ]:
p = 16
print(f"Selecting {p} optimal sensors...")
sensor_indices = qdeim_sensor_placement(Phi, p, num_channels=C)

sensor_x = mesh_X.flatten()[sensor_indices]
sensor_y = mesh_Y.flatten()[sensor_indices]

plt.figure(figsize=(10, 4))
plt.scatter(sensor_x, sensor_y, c='red', s=50, marker='x', label='Sensors')
plt.contourf(mesh_X, mesh_Y, X_train[0, 0], levels=50, cmap='RdBu_r', alpha=0.6)
plt.colorbar(label='u-velocity')
plt.title('Q-DEIM Sensor Placement over Flow Field')
plt.legend()
plt.show()

### 4. Gappy POD Reconstruction
We reconstruct the full test fields from just the 16 sensor measurements.

In [ ]:
N_spatial = ny * nx
multi_channel_indices = np.concatenate([sensor_indices + c * N_spatial for c in range(C)])
y_s_test = X_test_flat[multi_channel_indices, :]

print("Reconstructing with Gappy POD...")
a_gappy, x_hat_gappy = gappy_pod_reconstruct(y_s_test, multi_channel_indices, Phi[:, :r], mu=1e-3)

err_gappy = np.mean(relative_l2_error(X_test_flat.T, x_hat_gappy.T))
print(f"Average Relative L2 Error on Test Set: {err_gappy*100:.2f}%")

print("\nFor Deep Learning SciML Reconstruction (SensorMLP), run:")
print("  python scripts/run_full_experiments.py")